# 48. LCS with its published InstanceNorm

A control on notebook 39, where LCS with BatchNorm (`lcs_l_bn`) read
75.24 mean top-1, about 1.6 points under our US-Net A (76.86), and under A
at width 1.0 as well. Same protocol: CIFAR-100 32x32, CIFAR ResNet-50, 100
epochs, batch 256, SGD nesterov lr 0.2, wd 5e-4, widths 0.25-1.00.

| | GPU | method | code |
|---|---|---|---|
| lcs_l_in | 0 | LCS+L (Nunez et al., WACV 2023) in the authors' published configuration: the line-subspace InstanceNorm `LinesAdaptiveIN`. Variant (a) of `lcs_README.md`: the flags of lcs_l_bn without `--norm BN --recal_batches 20` | apple/learning-compressible-subspaces at e6d3924 + `third_party/patches/lcs.patch` |

One run: GPU 1 stays idle (T4 x2 is fine, one card is enough).

Why: lcs_l_bn normalises with `LinesAdaptiveBN`, a class we wrote in
`lcs.patch` (upstream has no BatchNorm line class). lcs_l_in uses only
upstream norm classes, and InstanceNorm keeps no running statistics, so
there is no recalibration either. If lcs_l_in lands near lcs_l_bn (75.24),
our BN class is fine and LCS's gap is the method (a summed cross-entropy
over four widths, no in-place distillation); if it is clearly higher, our
BN class is suspect.

## More than one session

InstanceNorm is about 2x slower than BatchNorm here: ~9.2 h on an RTX 5060
Ti (lcs_README.md, Timing), ~18-27 h on a T4, so 2-3 sessions. LCS writes
`ckpt/lcs_l_in/last.pt` after every epoch. When the session ends, Save
Version, then in a new run attach that version's output (pick the version
by number, not Latest) and run again: the notebook finds the checkpoint and
the run continues. It ends with the sixteen protocol lines. Attach the
CIFAR-100 dataset as well.

In [ ]:
# Fixed for this notebook.
REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'
# key: (repository, pinned commit, folder, patch, extra pip packages)
UPSTREAMS = {'lcs': ('https://github.com/apple/learning-compressible-subspaces.git', 'e6d3924368faccbdfd3d89c4a4735dba947275c9', 'learning-compressible-subspaces', 'lcs.patch', 'pyyaml')}
WORKER_ENV = {'JOSLIM_WORKERS': '2', 'LCS_WORKERS': '2'}

In [ ]:
import os, queue, re, shutil, subprocess, sys, threading, time
import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
if n_gpu < 1:
    raise SystemExit('Needs a GPU: the run uses GPU 0 only.')

WORK = '/kaggle/working'
OURS = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(OURS):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, OURS],
                   check=True)
print('our code at', subprocess.run(
    ['git', '-C', OURS, 'log', '-1', '--format=%h %s'],
    capture_output=True, text=True).stdout.strip())

CODE = {}
packages = []
for key, (url, commit, folder, patch, pip) in UPSTREAMS.items():
    CODE[key] = os.path.join(WORK, folder)
    if not os.path.isdir(CODE[key]):
        subprocess.run(['git', 'clone', url, CODE[key]], check=True)
        subprocess.run(['git', '-C', CODE[key], 'checkout', '-q', commit],
                       check=True)
        subprocess.run(['git', '-C', CODE[key], 'apply', os.path.join(
            OURS, 'third_party', 'patches', patch)], check=True)
    print(key, 'at', subprocess.run(
        ['git', '-C', CODE[key], 'log', '-1', '--format=%h %ad',
         '--date=short'], capture_output=True, text=True).stdout.strip(),
        '+', patch)
    packages += pip.split()

# install without letting pip replace Kaggle's torch
base = torch.__version__.split('+')[0]
with open(os.path.join(WORK, 'constraints.txt'), 'w') as handle:
    handle.write('torch=={}\n'.format(base))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c',
                os.path.join(WORK, 'constraints.txt')] + packages, check=True)

In [ ]:
# the folder that contains cifar-100-python/, or a place to download it
DATA = None
for root, dirs, _ in os.walk('/kaggle/input', followlinks=True):
    if 'cifar-100-python' in dirs:
        DATA = root
        break
if DATA is None:
    DATA = os.path.join(WORK, 'data')
    os.makedirs(DATA, exist_ok=True)
    print('no CIFAR-100 attached: it will be downloaded into', DATA)
else:
    print('CIFAR-100 from', DATA)

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs):
    """one command per card, output tagged and interleaved"""
    lines = queue.Queue()
    procs = {}

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for label, gpu, cwd, command in jobs:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, **WORKER_ENV)
        proc = subprocess.Popen(command, cwd=cwd, env=env, text=True,
                                stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] gpu {}: {}'.format(label, gpu, ' '.join(command)),
              flush=True)
    started, remaining = time.time(), len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[round(float(width), 2)] = (
                float(loss), float(top1))
        print('[{}] {}'.format(label, line), flush=True)
    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
CK = os.path.join(WORK, 'ckpt')
# LCS keeps ckpt/lcs_l_in/last.pt
wanted = {'lcs_l_in': ('last.pt', 'lcs_l_in')}
for label, (name, folder) in wanted.items():
    target = os.path.join(CK, folder or '', name)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        for root, _, files in os.walk('/kaggle/input', followlinks=True):
            if name in files and (folder is None
                                  or os.path.basename(root) == folder):
                shutil.copy(os.path.join(root, name), target)
                print('restored', label, 'from', root)
                break
    print(label, 'resumes from its checkpoint' if os.path.exists(target)
          else 'starts at epoch 1')

In [ ]:
WIDTHS = ','.join('{:.2f}'.format(0.25 + 0.05 * i) for i in range(16))
LCS = ['--model', 'cresnet50', '--dataset', 'cifar100', '--method', 'lcs_l',
       '--data_dir', DATA, '--epochs', '100', '--batch_size', '256',
       '--learning_rate', '0.2', '--momentum', '0.9', '--nesterov',
       '--weight_decay', '5e-4', '--width_factor_limits', '0.25,1.0',
       '--eval_width_factors', WIDTHS, '--skip_upstream_test',
       '--save_dir', os.path.join(WORK, 'lcs_l_in'),
       '--ckpt_dir', os.path.join(CK, 'lcs_l_in'), '--log_prefix', 'lcs_l_in']
# LCS ends its run with the sixteen protocol lines
codes = run_pinned([
    ('lcs_l_in', '0', CODE['lcs'],
     [sys.executable, '-u', 'train_structured.py'] + LCS)])
print(codes)

## Results, against A (BX, 76.86) and K (DD, 77.50)

Send back this table and the `val ... -1/100` lines above it.

How to read lcs_l_in: near lcs_l_bn (75.24 mean) means our BatchNorm
class is fine and LCS's gap to A is the method; clearly higher means our
BatchNorm class is suspect.

In [ ]:
A_REF = {0.25: 75.26, 0.3: 75.35, 0.35: 75.54, 0.4: 76.01, 0.45: 76.3, 0.5: 76.77, 0.55: 76.63, 0.6: 77.03, 0.65: 77.13, 0.7: 77.2, 0.75: 77.39, 0.8: 77.56, 0.85: 77.6, 0.9: 77.99, 0.95: 78.01, 1.0: 78.04}
K_REF = {0.25: 75.63, 0.3: 76.29, 0.35: 76.54, 0.4: 76.97, 0.45: 77.67, 0.5: 77.43, 0.55: 77.58, 0.6: 77.62, 0.65: 77.83, 0.7: 78.02, 0.75: 78.0, 0.8: 78.01, 0.85: 78.09, 0.9: 78.23, 0.95: 78.08, 1.0: 77.95}
labels = ['lcs_l_in']
print('{:>7}{:>8}{:>8}'.format('width', 'A', 'K') + ''.join(
    '{:>12}{:>7}{:>7}'.format(b[:11], 'vs A', 'vs K') for b in labels))
for width in sorted(A_REF):
    row = '{:>7.2f}{:>8.2f}{:>8.2f}'.format(
        width, A_REF[width], K_REF[width])
    for label in labels:
        entry = results.get(label, {}).get(width)
        if entry is None:
            row += '{:>12}{:>7}{:>7}'.format('-', '-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>12.2f}{:>+7.2f}{:>+7.2f}'.format(
            accuracy, accuracy - A_REF[width], accuracy - K_REF[width])
    print(row)
reference = sum(A_REF.values()) / len(A_REF)
k_reference = sum(K_REF.values()) / len(K_REF)
print('\n{:24} mean {:.2f}'.format('A (BX)', reference))
print('{:24} mean {:.2f}'.format('K (DD)', k_reference))
for label in labels:
    table = results.get(label, {})
    if len(table) == len(A_REF):
        mean = sum(100.0 * (1.0 - v[1]) for v in table.values()) / len(table)
        print('{:24} mean {:.2f}   vs A {:+.2f}   vs K {:+.2f}'.format(
            label, mean, mean - reference, mean - k_reference))
    else:
        print('{:24} {} of 16 widths read'.format(label, len(table)))

In [ ]:
# keep the checkpoints in the output, under a name the next session finds
print('checkpoints in', CK, sorted(os.listdir(CK)))